<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkST0151ENSkillsNetwork20531532-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>



# **Final Project: Boston Housing**


##### Estimated time needed: **60** minutes


#### Import the required libraries we need for the lab.


In [ ]:
import piplite

await piplite.install(['numpy', 'pandas'])
await piplite.install(['seaborn'])
await piplite.install(['scipy'])
await piplite.install(['statsmodels'])

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

sns.set_theme(style="whitegrid")

#### Read the dataset in the csv file from the URL


In [ ]:
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/boston_housing.csv'
resp = await fetch(URL)
boston_url = io.BytesIO((await resp.arrayBuffer()).to_py())

In [ ]:
boston_df=pd.read_csv(boston_url)

#### Add your code below following the instructions given in the course to complete the peer graded assignment


In [ ]:
boston_df.head()

In [ ]:
boston_df.dtypes

In [ ]:
boston_df.isnull().sum()

In [ ]:
print("Total missing values:", boston_df.isnull().sum().sum())

In [ ]:
print("Number of duplicate rows:", boston_df.duplicated().sum())

In [ ]:
correlation_matrix = boston_df.corr(numeric_only=True)

display(correlation_matrix)

plt.figure(figsize=(12, 9))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Matrix of Boston Housing Dataset")
plt.tight_layout()
plt.show()

In [ ]:
boston_df.describe()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(y=boston_df["MEDV"])

plt.title("Boxplot of Median House Values (MEDV)")
plt.ylabel("MEDV (in $1,000s)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(data=boston_df, x="CHAS")

plt.title("Number of Properties Bounded by the Charles River")
plt.xlabel("CHAS (0 = No, 1 = Yes)")
plt.ylabel("Number of Properties")
plt.tight_layout()
plt.show()

In [ ]:
boston_df["AGE_GROUP"] = pd.qcut(
    boston_df["AGE"],
    q=3,
    labels=["Low AGE", "Medium AGE", "High AGE"],
    duplicates="drop"
)

plt.figure(figsize=(9, 6))

sns.boxplot(
    data=boston_df,
    x="AGE_GROUP",
    y="MEDV",
    order=["Low AGE", "Medium AGE", "High AGE"]
)

plt.title("Median House Values by AGE Group")
plt.xlabel("AGE Group")
plt.ylabel("MEDV (in $1,000s)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=boston_df,
    x="INDUS",
    y="NOX",
    alpha=0.7
)

plt.title("Relationship Between INDUS and NOX")
plt.xlabel("INDUS (Non-retail Business Acres per Town)")
plt.ylabel("NOX (Nitric Oxide Concentration)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=boston_df,
    x="PTRATIO",
    bins=20,
    kde=True
)

plt.title("Distribution of Pupil-Teacher Ratio (PTRATIO)")
plt.xlabel("Pupil-Teacher Ratio")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
chas_0 = boston_df.loc[boston_df["CHAS"] == 0, "MEDV"].dropna()
chas_1 = boston_df.loc[boston_df["CHAS"] == 1, "MEDV"].dropna()

levene_stat, levene_p = stats.levene(chas_0, chas_1)

print("Levene's Test")
print("Test statistic:", levene_stat)
print("P-value:", levene_p)

if levene_p < 0.05:
    print("Conclusion: Reject the null hypothesis. The variances differ significantly.")
else:
    print("Conclusion: Fail to reject the null hypothesis. There is insufficient evidence that the variances differ.")

In [ ]:
# Use Welch's t-test when Levene's test indicates unequal variances.
# Otherwise, use the standard independent samples t-test.

t_stat, t_p = stats.ttest_ind(
    chas_0,
    chas_1,
    equal_var=(levene_p >= 0.05)
)

print("Independent Samples T-Test")
print("Mean MEDV for CHAS = 0:", chas_0.mean())
print("Mean MEDV for CHAS = 1:", chas_1.mean())
print("T-statistic:", t_stat)
print("P-value:", t_p)

if t_p < 0.05:
    print("Conclusion: Reject the null hypothesis. The mean MEDV differs significantly between the two groups.")
else:
    print("Conclusion: Fail to reject the null hypothesis. There is insufficient evidence of a difference in mean MEDV.")

In [ ]:
age_model = ols(
    "MEDV ~ C(AGE_GROUP)",
    data=boston_df
).fit()

anova_results = sm.stats.anova_lm(age_model, typ=2)

print(anova_results)

anova_p = anova_results.loc["C(AGE_GROUP)", "PR(>F)"]

if anova_p < 0.05:
    print("Conclusion: Reject the null hypothesis. At least one AGE group has a significantly different mean MEDV.")
else:
    print("Conclusion: Fail to reject the null hypothesis. There is insufficient evidence of a difference in mean MEDV across the AGE groups.")

In [ ]:
correlation, correlation_p = stats.pearsonr(
    boston_df["NOX"],
    boston_df["INDUS"]
)

print("Pearson Correlation Test")
print("Correlation coefficient:", correlation)
print("P-value:", correlation_p)

if correlation_p < 0.05:
    print("Conclusion: Reject the null hypothesis. There is a statistically significant linear correlation.")
else:
    print("Conclusion: Fail to reject the null hypothesis. There is insufficient evidence of a linear correlation.")

In [ ]:
regression_model = ols(
    "MEDV ~ DIS",
    data=boston_df
).fit()

print(regression_model.summary())

In [ ]:
intercept = regression_model.params["Intercept"]
slope = regression_model.params["DIS"]
r_squared = regression_model.rsquared
regression_p = regression_model.pvalues["DIS"]

print("Regression Equation:")
print(f"MEDV = {intercept:.4f} + ({slope:.4f} * DIS)")

print("\nRegression Results")
print("Intercept:", intercept)
print("DIS coefficient:", slope)
print("R-squared:", r_squared)
print("P-value for DIS:", regression_p)

if regression_p < 0.05:
    print("Conclusion: DIS is a statistically significant linear predictor of MEDV.")
else:
    print("Conclusion: There is insufficient evidence that DIS is a significant linear predictor of MEDV.")

In [ ]:
plt.figure(figsize=(8, 6))

sns.regplot(
    data=boston_df,
    x="DIS",
    y="MEDV",
    scatter_kws={"alpha": 0.6},
    line_kws={"color": "red"}
)

plt.title("Linear Regression: DIS vs MEDV")
plt.xlabel("DIS (Weighted Distance to Employment Centres)")
plt.ylabel("MEDV (in $1,000s)")
plt.tight_layout()
plt.show()